# Feature engineering
Building model inputs: joins, windows and aggregates stacked together, then summarised.

This is the closest thing here to a real pipeline rather than a single query, and it is the
section most relevant to anyone doing data science work in notebooks.

**F2** is the most common feature family in existence — spend in the last 30 and 90 days per
customer. **F3** is target-encoding style: compute a group average, then join it back onto
every row.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from bench import config as C, datagen, engines, report
from bench.harness import Case, Bench

paths = datagen.paths(C.MAIN_SIZE)
duck  = engines.get_duckdb()
spark = engines.get_spark()
engines.attach(duck, spark, paths)
bench = Bench(duck, spark, "07_features", C.MAIN_SIZE)
print(f"Ready. {C.human(C.MAIN_SIZE)} sales rows. "
      f"Both engines have {C.ENGINE_MEMORY_MB} MB and {C.plural(C.ENGINE_THREADS, 'thread')}.")

In [ ]:
SQL_F1 = """
SELECT segment, count(*) AS rows_built,
       round(avg(spend_so_far),2) AS avg_spend, round(avg(recent_avg),2) AS avg_recent,
       max(purchase_no) AS max_purchases
FROM (
  SELECT c.segment,
    sum(s.amount) OVER (PARTITION BY s.customer_id ORDER BY s.sale_ts) AS spend_so_far,
    avg(s.amount) OVER (PARTITION BY s.customer_id ORDER BY s.sale_ts
                        ROWS BETWEEN 4 PRECEDING AND CURRENT ROW)      AS recent_avg,
    row_number() OVER (PARTITION BY s.customer_id ORDER BY s.sale_ts)  AS purchase_no
  FROM sales s
  JOIN customers c ON c.customer_id = s.customer_id
  JOIN products  p ON p.product_id  = s.product_id
  WHERE s.amount IS NOT NULL
) t GROUP BY 1 ORDER BY 1
"""

# WHY THIS ONE:
#   Joins plus windows plus aggregates — a realistic feature pipeline.
_, out, _ = bench.run(Case("F1", "Wide feature table", "Features", sql=SQL_F1.strip(),
                          why="Joins plus windows plus aggregates — a realistic feature pipeline."))
display(out.head())

In [ ]:
SQL_F2 = """
SELECT count(*) AS customers, round(avg(spend_30d),2) AS avg_30d,
       round(avg(spend_90d),2) AS avg_90d
FROM (
  SELECT customer_id,
    sum(CASE WHEN sale_date >= DATE '2025-04-01' THEN amount ELSE 0 END) AS spend_30d,
    sum(CASE WHEN sale_date >= DATE '2025-02-01' THEN amount ELSE 0 END) AS spend_90d
  FROM sales WHERE customer_id IS NOT NULL GROUP BY 1
) t
"""

# WHY THIS ONE:
#   Last 30 / 90 days per customer — the most common feature family there is.
_, out, _ = bench.run(Case("F2", "Time-window aggregates per customer", "Features", sql=SQL_F2.strip(),
                          why="Last 30 / 90 days per customer — the most common feature family there is."))
display(out.head())

In [ ]:
SQL_F3 = """
SELECT count(*) AS n, round(avg(ratio_to_category),4) AS avg_ratio FROM (
  SELECT s.amount / NULLIF(g.avg_amount, 0) AS ratio_to_category
  FROM sales s
  JOIN products p ON p.product_id = s.product_id
  JOIN (SELECT p2.category, avg(s2.amount) AS avg_amount
        FROM sales s2 JOIN products p2 ON p2.product_id = s2.product_id
        GROUP BY 1) g ON g.category = p.category
  WHERE s.amount IS NOT NULL
) t
"""

# WHY THIS ONE:
#   Target-encoding style: compute a group average, then join it back to every row.
_, out, _ = bench.run(Case("F3", "Group statistics joined back to rows", "Features", sql=SQL_F3.strip(),
                          why="Target-encoding style: compute a group average, then join it back to every row."))
display(out.head())

In [ ]:
SQL_F4 = """
SELECT count(*) AS customers, round(avg(frequency),2) AS avg_freq,
       round(avg(monetary),2) AS avg_monetary
FROM (
  SELECT customer_id, max(sale_date) AS recency, count(*) AS frequency,
         sum(COALESCE(amount,0)) AS monetary
  FROM sales WHERE customer_id IS NOT NULL GROUP BY 1
) t
"""

# WHY THIS ONE:
#   Recency / frequency / money — the standard customer feature set.
_, out, _ = bench.run(Case("F4", "Customer RFM summary", "Features", sql=SQL_F4.strip(),
                          why="Recency / frequency / money — the standard customer feature set."))
display(out.head())

## Results for this notebook

`Same SQL?` tells you whether both engines ran the *identical* SQL string. Where it says no, the two dialects genuinely differ and the case is written twice.

`Same answer?` is the check that matters: a fast wrong answer is worth nothing.

In [ ]:
report.headline(bench.table())
print()
display(report.results_table(bench.table()))
report.times_chart(bench.table())
bench.save()
engines.stop_spark()